# Team Zero — Scam Message Checker: the word-based model

**Problem:** scam SMS and chat messages trick people into paying fake fees or sharing bank details,
and most people cannot tell a scam from a genuine message until the money is gone.

**This notebook** trains the first layer of our checker: a word-based ML model (TF-IDF + logistic regression)
that scores how scam-like a message's wording is. The other layers (scam tactics, payment-details warning,
link checks, receiver check, Granite explanation) live in `checker/` and the API.

It runs on **IBM LinuxONE** (s390x) inside IBM's Jupyter container.

## 1. Where this runs

In [ ]:
import platform
import sys

import pandas as pd
import sklearn

print("Machine:", platform.machine())  # s390x = IBM Z / LinuxONE
print("OS:", platform.system(), platform.release())
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__, "| scikit-learn:", sklearn.__version__)

## 2. Load the data

UCI **SMS Spam Collection** (Almeida & Gómez Hidalgo, CC BY 4.0): 5,574 English SMS labelled `ham` or `spam`.
The file is tab-separated with no header row. `QUOTE_NONE` is needed because some messages contain `"`;
without it pandas merges lines and reads 5,572 rows.

In [ ]:
import csv

pd.set_option("display.max_colwidth", None)

df = pd.read_csv(
    "data/sms_spam_collection/SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"],
    quoting=csv.QUOTE_NONE,
)
print(df.shape)
df.head()

## 3. Explore

In [ ]:
print("Empty values per column:")
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True).round(3))

In [ ]:
df["length"] = df["message"].str.len()
df.groupby("label")["length"].mean().round(1)

In [ ]:
import re
from collections import Counter

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


def top_words(label, n=15):
    words = Counter()
    for m in df[df["label"] == label]["message"]:
        for w in re.findall(r"[a-z]+", m.lower()):
            if w not in ENGLISH_STOP_WORDS and len(w) > 1:
                words[w] += 1
    return words.most_common(n)


print("SPAM:", top_words("spam"))
print("HAM: ", top_words("ham"))

In [ ]:
import matplotlib.pyplot as plt

df[df["label"] == "ham"]["length"].plot.hist(bins=range(0, 300, 5), alpha=0.6, label="ham")
df[df["label"] == "spam"]["length"].plot.hist(bins=range(0, 300, 5), alpha=0.6, label="spam")
plt.xlabel("Message length (characters)")
plt.title("Spam messages are longer: most fill a whole SMS (160 characters)")
plt.legend()
plt.show()

## 4. Prepare

Drop duplicates (so the same message cannot end up in both training and test data), separate the input
(`X`, the message) from the answer (`y`: 1 = spam, 0 = ham), and keep 20% of the messages aside for testing.
`stratify=y` keeps the same ham/spam mix in both parts.

In [ ]:
from sklearn.model_selection import train_test_split

df = df.drop_duplicates()
X = df["message"]
y = (df["label"] == "spam").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Unique messages:", len(df))
print("Train:", len(X_train), "| Test:", len(X_test))
print("Spam share - train:", round(y_train.mean(), 3), "| test:", round(y_test.mean(), 3))

## 5. Model 1: TF-IDF + logistic regression

TF-IDF turns each message into one score per word (high for words that are frequent in the message and rare
overall). Logistic regression learns one weight per word and turns the sum into a probability.
`class_weight="balanced"`: spam is only 13% of the data, so mistakes on spam count more during training.
Both steps are kept together in one **pipeline**, so the saved model takes plain text.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.pipeline import Pipeline

logreg = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
logreg.fit(X_train, y_train)
logreg_pred = logreg.predict(X_test)

print(classification_report(y_test, logreg_pred, target_names=["ham", "spam"], digits=3))
ConfusionMatrixDisplay.from_predictions(y_test, logreg_pred, display_labels=["ham", "spam"])
plt.title("Logistic regression on the test set")
plt.show()

## 6. Model 2: Naive Bayes (for comparison)

A classic text model that counts how often each word appears in spam versus ham.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nbayes = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("model", MultinomialNB()),
])
nbayes.fit(X_train, y_train)
nbayes_pred = nbayes.predict(X_test)

print(classification_report(y_test, nbayes_pred, target_names=["ham", "spam"], digits=3))
ConfusionMatrixDisplay.from_predictions(y_test, nbayes_pred, display_labels=["ham", "spam"])
plt.title("Naive Bayes on the test set")
plt.show()

## 7. Compare and choose

**Precision** (spam): when the model says spam, how often is it right?
**Recall** (spam): of all real spam, how much did it catch? For a scam checker, missing a scam is the worse mistake,
so we pick the model with the better **F1** (the balance of both) and look closely at recall.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

results = pd.DataFrame(
    {
        name: {
            "accuracy": accuracy_score(y_test, pred),
            "precision (spam)": precision_score(y_test, pred),
            "recall (spam)": recall_score(y_test, pred),
            "F1 (spam)": f1_score(y_test, pred),
        }
        for name, pred in [("logistic regression", logreg_pred), ("naive bayes", nbayes_pred)]
    }
).round(3)
print(results)

best_name = results.loc["F1 (spam)"].idxmax()
best = logreg if best_name == "logistic regression" else nbayes
print("\nChosen model:", best_name)

## 8. Risk bands

The model gives a spam probability from 0 to 1. The checker shows it as a band:
**Safe** below 0.30, **Suspicious** from 0.30 to below 0.70, **Scam** from 0.70.
The same thresholds are used by the API (`checker/model.py`), so the notebook and the app always agree.

In [ ]:
from checker.model import SAFE_BELOW, SCAM_FROM, band

test_prob = best.predict_proba(X_test)[:, 1]
bands = pd.Series([band(p) for p in test_prob], index=X_test.index, name="band")
truth = y_test.map({0: "ham", 1: "spam"}).rename("truth")

print(f"Safe < {SAFE_BELOW} <= Suspicious < {SCAM_FROM} <= Scam")
pd.crosstab(truth, bands)[["safe", "suspicious", "scam"]]

## 9. Our own test messages (modern scams, Malay)

These 20 messages are made up (created by the team with AI assistance) to cover cases the 2011 dataset lacks:
modern scams, Malay messages, and a normal-looking bill with a swapped bank account.
The word model alone misses several of them, which is why the checker adds rule-based layers on top.

In [ ]:
tests = pd.read_csv("data/test_messages/test_messages.csv")
tests["model_probability"] = best.predict_proba(tests["message"])[:, 1].round(3)
tests["model_band"] = [band(p) for p in tests["model_probability"]]
tests["match"] = tests["model_band"] == tests["expected_verdict"]

print("Word model matches the expected verdict for", tests["match"].sum(), "of", len(tests), "messages")
tests[["id", "language", "expected_verdict", "model_probability", "model_band", "match", "message"]]

## 10. Save the model

The whole pipeline (TF-IDF + model) is saved as one file in `models/`, then loaded back to check it gives
the same answers. The API loads this file. Model files are not committed to git: the model is trained here,
on LinuxONE.

In [ ]:
import os

import joblib

os.makedirs("models", exist_ok=True)
model_path = "models/scam_model.joblib"
joblib.dump(best, model_path)

reloaded = joblib.load(model_path)
same = (reloaded.predict_proba(X_test)[:, 1] == best.predict_proba(X_test)[:, 1]).all()
print("Saved:", model_path, "| size:", os.path.getsize(model_path), "bytes | reload gives same answers:", same)